## Global instructions

Only three input files are required:
1) a project-specific amino-acid sequence CSV
2) `overhangs.csv`
3) `orthogonal_oligos.csv`

The AAseq CSV should be your variant names and their associated amino acid sequences. Make sure all genes have unique names.

For each run, edit the `User inputs` code cells for each section, starting with `AA_INPUT_PATH`. By default, every generated file is written beside that AA-sequence CSV. `RUN_NAME=None` uses the project folder name (for example `opTF009`), and `RUN_OUTPUT_DIR=None` writes beside the input file.

The reusable `overhangs.csv` and `orthogonal_oligos.csv` inventories stay in the repository `data/` folder. Each run writes its used-overhang list and remaining unused oligos into the project folder for traceability.

`overhangs.csv` defines the set of high-fidelity Golden Gate overhangs to use. I use NEB GETset (https://ligasefidelity.neb.com/getset/run.cgi) and enter my required vector overhangs (GCTT and AGTG) to retrieve overhang sets compatible with those. I typically use 32 but more or less can also be used. 

orthogonal_oligos.csv contains the set or primers to use for amplification in 5'-3' orientation and will be used to add those binding sites to each sub-pool uniquely


In Gene Optimisation certain variables can be defined e.g. restriction sites to avoid, homopolymers to avoid, GC content, host to optimise for etc. all defined by the DNAchisel tool (https://edinburgh-genome-foundry.github.io/DnaChisel/index.html). This step can also be skipped if optimised elsewhere and supplied to Pool Assignment in the file Optimised_genes.csv

In Pool Assignment certain variables can be defined e.g. length of oPool (I typically use 300 or 250nt), the defined vector overhangs (to check they are not in your overhang list to assign to pools) and the maximum size of a 'short' sub-pool i.e. genes requiring only one fragment for assembly. 

In Primer and BsaI site addition certain variable can be defined e.g. Vector overhangs, the restriction enzyme site (including an A,G,C or T for its N e.g. BsaI is GGTCTCN| so I pass it GGTCTCA"). This will output three files, one FULL_INFO.csv with all information on pool assignment, primers to use, number of fragments etc., one references.fasta containing all gene sequences, and one oPool_Order_Fragments.csv which can be uploaded directly to Twist for ordering! 




## Gene optimisation

### User inputs

Edit only the next cell for a new project, then run it. Paths can be pasted as ordinary quoted text. Leave an optional value as `None` to use the documented default.

In [ ]:
# ============================================================
# USER INPUTS - edit only this cell for a new project
# ============================================================

# REQUIRED: CSV containing sequence names and amino-acid sequences.
# Outputs are written into the same folder as this file by default.
AA_INPUT_PATH = "data/AAseq_dTF001_dTF016.csv"

# OPTIONAL: output filename prefix. None uses the input file's folder name (for example opTF009).
RUN_NAME = None

# OPTIONAL: output folder. None writes outputs beside AA_INPUT_PATH.
RUN_OUTPUT_DIR = None

# Reusable repository input files. A filename alone is looked up in oPool_Optimiser/data/.
OVERHANGS_PATH = "overhangs.csv"
ORTHOGONAL_OLIGOS_PATH = "orthogonal_oligos.csv"

# Type IIS recognition sequence only; do not include the separate N base.
# Its reverse complement is avoided automatically everywhere.
TYPEIIS_RECOGNITION_SITE = "GGTCTC"  # BsaI

# Additional motifs to avoid during gene optimization.
# The Type IIS site and its reverse complement are added automatically.
AVOID_SEQS = [
    "AAAAA",
    "GGGGG",
    "CCCCC",
    "TTTTT",
    # "GAAGAC",  # example: BbsI
]

# Allowed GC fraction (0.30 means 30%) and the window size used to check it.
GC_MIN = 0.30
GC_MAX = 0.70
GC_WINDOW = 50

# Codon-optimization host and method.
# Built-in species keywords: b_subtilis, c_elegans, d_melanogaster, e_coli,
# g_gallus, h_sapiens, m_musculus, m_musculus_domesticus, s_cerevisiae.
# A numeric NCBI taxonomy ID is also accepted but requires internet access.
CODON_SPECIES = "e_coli"
CODON_METHOD = "match_codon_usage"


In [ ]:
# SETUP CODE - run this cell after User inputs; you should not need to edit it.

import pandas as pd
from pathlib import Path
from Bio.Seq import Seq
from Bio.Data import CodonTable
from dnachisel import (
    DnaOptimizationProblem,
    AvoidPattern,
    EnforceGCContent,
    EnforceTranslation,
    CodonOptimize,
)

# ============================================================
# PROJECT SETUP - usually do not edit
# ============================================================

def find_project_root() -> Path:
    """Find the repo root whether Jupyter starts in the repo, notebook folder, or parent folder."""
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / "oPool_Optimiser"):
            if (candidate / "data" / "overhangs.csv").exists() and (candidate / "notebooks").exists():
                return candidate
    raise RuntimeError("Could not find the oPool_Optimiser repo root from the current working directory.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"


# ============================================================
# NORMALIZE AND VALIDATE USER INPUTS - usually do not edit
# ============================================================

TYPEIIS_RECOGNITION_SITE = TYPEIIS_RECOGNITION_SITE.strip().upper()
if len(TYPEIIS_RECOGNITION_SITE) < 5 or any(base not in "ACGT" for base in TYPEIIS_RECOGNITION_SITE):
    raise ValueError("TYPEIIS_RECOGNITION_SITE must be at least 5 bases and contain only A/C/G/T.")
TYPEIIS_RECOGNITION_SITE_RC = str(Seq(TYPEIIS_RECOGNITION_SITE).reverse_complement())
TYPEIIS_FORBIDDEN_SITES = {TYPEIIS_RECOGNITION_SITE, TYPEIIS_RECOGNITION_SITE_RC}

_additional_avoid_seqs = [str(seq).strip().upper() for seq in AVOID_SEQS]
if any(not seq or any(base not in "ACGT" for base in seq) for seq in _additional_avoid_seqs):
    raise ValueError("Every entry in AVOID_SEQS must contain only A/C/G/T.")
AVOID_SEQS = list(dict.fromkeys([*sorted(TYPEIIS_FORBIDDEN_SITES), *_additional_avoid_seqs]))


# ============================================================
# DERIVED PATHS - downstream cells use these
# ============================================================

def _resolve_path(path_value, relative_to: Path) -> Path:
    """Resolve an absolute path or a path relative to a documented base folder."""
    path = Path(path_value).expanduser()
    return path.resolve() if path.is_absolute() else (relative_to / path).resolve()


def _default_run_name(aa_input: Path) -> str:
    """Use the project folder name, except for example inputs stored directly in repo data/."""
    if aa_input.parent.resolve() == DATA_DIR.resolve():
        return aa_input.stem
    return aa_input.parent.name


def build_run_paths(aa_input, run_name=None, run_output_dir=None, overhangs=None, primers=None):
    aa_input = _resolve_path(aa_input, PROJECT_ROOT)
    overhangs = _resolve_path(overhangs, DATA_DIR)
    primers = _resolve_path(primers, DATA_DIR)

    if run_output_dir is None or str(run_output_dir).strip() == "":
        run_dir = PROJECT_ROOT / "outputs" if aa_input.parent.resolve() == DATA_DIR.resolve() else aa_input.parent
    else:
        run_dir = _resolve_path(run_output_dir, aa_input.parent)

    if run_name is None or str(run_name).strip() == "":
        run_name = _default_run_name(aa_input)
    else:
        run_name = str(run_name).strip()

    optimised_path = run_dir / f"{run_name}_Optimised.csv"

    return {
        "run_name": run_name,
        "run_dir": run_dir,
        "aa_input": aa_input,
        "overhangs": overhangs,
        "primers": primers,
        "optimised": optimised_path,
        "assigned": run_dir / f"{run_name}_Assigned.csv",
        "strip_log": run_dir / f"{run_name}_stripped_ATG_log.csv",
        "unassigned": run_dir / f"{run_name}_unassigned.csv",
        "overhangs_used": run_dir / f"{run_name}_overhangs_used.csv",
        "full_info": run_dir / f"{run_name}_FULL_INFO.csv",
        "references_fasta": run_dir / f"{run_name}_references.fasta",
        "fragments": run_dir / f"{run_name}_oPool_Order_Fragments.csv",
        "merged_fragments": run_dir / f"{run_name}_all_oPool_Order_Fragments.csv",
        "full_info_merged": run_dir / f"{run_name}_FULL_INFO_merged.csv",
        "all_pairs": run_dir / f"{run_name}_orthogonal_oligos_pairs_all.csv",
        "unused_pairs": run_dir / f"{run_name}_orthogonal_oligos_pairs_unused.csv",
        "unused_primers": run_dir / f"{run_name}_orthogonal_oligos_unused.csv",
    }


RUN_PATHS = build_run_paths(
    AA_INPUT_PATH,
    run_name=RUN_NAME,
    run_output_dir=RUN_OUTPUT_DIR,
    overhangs=OVERHANGS_PATH,
    primers=ORTHOGONAL_OLIGOS_PATH,
)
RUN_NAME = RUN_PATHS["run_name"]
RUN_DIR = RUN_PATHS["run_dir"]
RUN_DIR.mkdir(parents=True, exist_ok=True)

for label, path_key in (
    ("AA input", "aa_input"),
    ("Overhang inventory", "overhangs"),
    ("Orthogonal-oligo inventory", "primers"),
):
    if not RUN_PATHS[path_key].is_file():
        raise FileNotFoundError(f"{label} file not found: {RUN_PATHS[path_key]}")

INPUT_PDB = RUN_NAME
input_path = RUN_PATHS["aa_input"]
output_path = RUN_PATHS["optimised"]

print(f"[PATHS] Run name: {RUN_NAME}")
print(f"[PATHS] AA input: {input_path}")
print(f"[PATHS] Output folder: {RUN_DIR}")
print(f"[PATHS] Overhang source: {RUN_PATHS['overhangs']}")
print(f"[PATHS] Oligo source: {RUN_PATHS['primers']}")

input_df = pd.read_csv(input_path, names=["name", "aa_seq"])

input_df

### Script

#### Reverse translates amino acid sequences into DNA

In [ ]:
def reverse_translate(protein_sequence: str) -> str:
    """
    Reverse translate an amino acid sequence to a DNA sequence using the standard codon table.
    Chooses the first available codon for each amino acid.

    Args:
        protein_sequence (str): Amino acid sequence using 1-letter codes.

    Returns:
        str: A DNA sequence (string of A, T, G, C).
    """
    # Get the standard codon table
    codon_table = CodonTable.unambiguous_dna_by_id[1]

    # Create a dictionary mapping amino acids to one possible codon
    aa_to_codon = {}
    for codon, aa in codon_table.forward_table.items():
        if aa not in aa_to_codon:
            aa_to_codon[aa] = codon.upper()

    # Add stop codon
    aa_to_codon['*'] = codon_table.stop_codons[0].upper()

    # Translate the amino acid sequence
    dna_sequence = ''
    for aa in protein_sequence.upper():
        if aa not in aa_to_codon:
            raise ValueError(f"Invalid amino acid: {aa}")
        dna_sequence += aa_to_codon[aa]

    return dna_sequence


input_df["dna_seq_unrefined"] = input_df.aa_seq.apply(lambda x: reverse_translate(x))

#### Optimises DNA sequences

In [ ]:
def dna_chisel_remove_RS_and_codon_optimize(
    dna_seq: str,
    avoid_seqs: list[str] | None = None
) -> str:
    """
    Run DNA Chisel on `dna_seq`, removing any short motifs in `avoid_seqs`
    and then codon‐optimizing the entire open reading frame.

    If `avoid_seqs` is None, it uses the global AVOID_SEQS defined in Cell 1.
    """

    # Use the passed-in list or fall back to the global list
    patterns_to_avoid = avoid_seqs if (avoid_seqs is not None) else AVOID_SEQS

    seq_length = len(dna_seq)

    # Build one AvoidPattern constraint per motif in patterns_to_avoid
    avoid_constraints = [AvoidPattern(pat) for pat in patterns_to_avoid]

    problem = DnaOptimizationProblem(
        sequence=dna_seq,
        constraints=[
            *avoid_constraints,
            EnforceGCContent(mini=GC_MIN, maxi=GC_MAX, window=GC_WINDOW),
            EnforceTranslation(location=(0, seq_length)),
        ],
        objectives=[
            CodonOptimize(
                species=CODON_SPECIES,
                method=CODON_METHOD,
                location=(0, seq_length),
            )
        ]
    )

    # Solve and optimize
    problem.resolve_constraints()
    problem.optimize()

    return problem.sequence

input_df["dna_seq_optimized"] = input_df.dna_seq_unrefined.apply(lambda x: dna_chisel_remove_RS_and_codon_optimize(x))

input_df = input_df.drop(columns=["dna_seq_unrefined"])

input_df.to_csv(output_path, index=False)


## Pool Assignment — optimized for long genes

This version avoids the factorial overhang-permutation search used by the working notebook. It indexes every native or one-codon synonymous overhang once, then searches a position-ordered cut path while pruning choices that cannot fit the remaining sequence. Internal overhangs remain unique within each sub-pool.

Set `GENES_PER_SUBPOOL = 1` in the next cell to place exactly one gene in each sub-pool, or leave it as `None` to pack as many genes as the available unique overhangs permit.

### Why this is not parallel by default

The old bottleneck was algorithmic (`overhang_count P number_of_cuts`), not a lack of CPU cores. The dynamic search is normally faster than the startup cost of macOS/Jupyter worker processes and remains deterministic. Progress and timing are printed so unusually difficult datasets are visible immediately.


### Pool-assignment inputs


In [ ]:
import bisect
import math
import time
import pandas as pd
from pathlib import Path
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Data import CodonTable

# ============================================================
# POOL ASSIGNMENT USER INPUTS
# ============================================================

# Final ordered oligo length. Any integer greater than 62 is supported.
OPOOL_LENGTH = 350

# Maximum genes in each sub-pool/block.
# None = automatically use the capacity allowed by unique overhangs.
# 1 = put every gene in its own sub-pool.
GENES_PER_SUBPOOL = None

# Maximum number of fragments attempted for a gene.
MAX_FRAGMENTS = 32

# Print indexing, block-assignment, and runtime progress.
SHOW_POOL_PROGRESS = True

# Vector overhang modes:
#   "fixed"       -> use VECTOR_OVERHANG_1_FIXED for every gene
#   "ser_windows" -> assign a unique Ser-Ser VectorOH1 within each sub-pool
VECTOR_OH1_MODE = "fixed"
VECTOR_OVERHANG_1_FIXED = "TATG"
VECTOR_OVERHANG_2 = "GGAT"

# Fragment 1 loses two available bases for a Ser window-0 VectorOH1.
VEC1_WINDOW0_FRAG1_PENALTY_NT = 2

# Additional limit for single-fragment genes. The smaller of this and
# GENES_PER_SUBPOOL is used. Set None for no separate short-gene limit.
SHORT_POOL_MAX_SIZE = 1000

# Remove the initial methionine codon when present.
STRIP_NTERM_MET = True
WRITE_STRIP_LOG = True

# ============================================================
# DERIVED PATHS — do not edit
# ============================================================
if "RUN_PATHS" not in globals():
    raise RuntimeError("Run the Gene optimisation user-input and setup cells first.")

STRIP_LOG_CSV = RUN_PATHS["strip_log"]
OVERHANGS_FILE = RUN_PATHS["overhangs"]
OVERHANGS_USED_CSV = RUN_PATHS["overhangs_used"]
INPUT_FILE = RUN_PATHS["optimised"]
OUTPUT_FILE = RUN_PATHS["assigned"]
UNASSIGNED_LOG = RUN_PATHS["unassigned"]


### Fast assignment and validation code


In [ ]:
# ----------------------------
# GLOBAL SETUP
# ----------------------------
def load_overhangs(path):
    """Read the comma-separated 4-nt overhang inventory from the first CSV cell."""
    df = pd.read_csv(path, header=None)
    raw = str(df.iloc[0, 0])
    overhangs = [oh.strip().upper() for oh in raw.split(",") if oh.strip()]
    if not overhangs:
        raise ValueError(f"No overhangs found in {path}")
    if len(overhangs) != len(set(overhangs)):
        raise ValueError("The overhang inventory contains duplicates.")
    if any(len(oh) != 4 or any(base not in "ACGT" for base in oh) for oh in overhangs):
        raise ValueError("Every internal overhang must be exactly four A/C/G/T bases.")
    return overhangs


OVERHANGS_LIST = load_overhangs(OVERHANGS_FILE)
OVERHANG_RANK = {overhang: index for index, overhang in enumerate(OVERHANGS_LIST)}
OVERHANG_BITS = {overhang: 1 << index for index, overhang in enumerate(OVERHANGS_LIST)}

codon_table = CodonTable.unambiguous_dna_by_name["Standard"]
syn_map = {}
for codon, aa in codon_table.forward_table.items():
    syn_map.setdefault(aa, []).append(codon)
for aa in syn_map:
    syn_map[aa] = sorted(syn_map[aa])

SER_CODONS = ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"]


# ----------------------------
# Ser-window VectorOH1 sets
# ----------------------------
def build_ser_windows_vec1_sets():
    w0 = set()
    w2 = set()
    for c1 in SER_CODONS:
        for c2 in SER_CODONS:
            ser_ser = c1 + c2
            w0.add(ser_ser[0:4])
            w2.add(ser_ser[2:6])
    return w0, w2, w0 | w2


VEC1_W0_SET, VEC1_W2_SET, VEC1_SER_UNION = build_ser_windows_vec1_sets()
VECTOR_OH2 = VECTOR_OVERHANG_2.strip().upper()
VECTOR_OH1_FIXED = VECTOR_OVERHANG_1_FIXED.strip().upper()


def build_forbidden_internal_overhangs():
    forbidden = {VECTOR_OH2}
    if VECTOR_OH1_MODE == "fixed":
        forbidden.add(VECTOR_OH1_FIXED)
    elif VECTOR_OH1_MODE == "ser_windows":
        forbidden |= VEC1_SER_UNION
    else:
        raise ValueError("VECTOR_OH1_MODE must be 'fixed' or 'ser_windows'.")
    return forbidden


FORBIDDEN_INTERNAL = build_forbidden_internal_overhangs()
INTERNAL_OVERHANGS_LIST = [
    overhang for overhang in OVERHANGS_LIST
    if overhang not in FORBIDDEN_INTERNAL
]
INTERNAL_OVERHANG_SET = set(INTERNAL_OVERHANGS_LIST)


# ----------------------------
# Input/output helpers
# ----------------------------
def load_sequences(path):
    """Load optimized DNA records and optionally strip a leading ATG."""
    df = pd.read_csv(path, dtype={"name": str})
    required = {"name", "dna_seq_optimized"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")

    records = []
    strip_log = []
    for _, row in df.iterrows():
        gid = str(row["name"]).strip()
        seq = str(row["dna_seq_optimized"]).strip().upper()
        if not gid or not seq:
            continue
        if any(base not in "ACGT" for base in seq):
            raise ValueError(f"Sequence {gid!r} contains a non-ACGT character.")
        if len(seq) % 3 != 0:
            raise ValueError(f"Sequence {gid!r} has length {len(seq)}, which is not divisible by 3.")

        original_len = len(seq)
        if STRIP_NTERM_MET and seq.startswith("ATG"):
            seq = seq[3:]
            strip_log.append({
                "Sequence Name": gid,
                "Action": "Removed leading ATG",
                "Original_Length": original_len,
                "New_Length": len(seq),
                "Original_Start": "ATG",
                "New_Start": seq[:12],
            })
        records.append(SeqRecord(Seq(seq), id=gid, description=""))

    if len({record.id for record in records}) != len(records):
        raise ValueError("Sequence names must be unique.")
    return records, strip_log


def write_unassigned_log(unassigned_ids, csv_path):
    pd.DataFrame({"Sequence Name": unassigned_ids}).to_csv(csv_path, index=False)


def write_used_overhangs(df, csv_path):
    """Write unique internal overhangs used anywhere in the run."""
    overhang_cols = [column for column in df.columns if column.startswith("Overhang")]
    used_set = {
        str(value).strip().upper()
        for column in overhang_cols
        for value in df[column]
        if pd.notna(value) and str(value).strip().upper() not in {"", "N/A"}
    }
    used_ordered = [overhang for overhang in OVERHANGS_LIST if overhang in used_set]
    used_ordered.extend(sorted(used_set - set(used_ordered)))
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([[",".join(used_ordered)]]).to_csv(csv_path, index=False, header=False)
    return used_ordered


# ----------------------------
# Fast synonymous-cut indexing
# ----------------------------
def contains_forbidden_typeiis_site(sequence):
    sequence = str(sequence).strip().upper()
    return any(site in sequence for site in TYPEIIS_FORBIDDEN_SITES)


def _mutated_slice(sequence, start, end, codon_start, new_codon):
    """Return sequence[start:end] after one proposed codon replacement."""
    chars = list(sequence[start:end])
    for position in range(max(start, codon_start), min(end, codon_start + 3)):
        chars[position - start] = new_codon[position - codon_start]
    return "".join(chars)


def _mutation_is_typeiis_safe(sequence, codon_start, new_codon):
    """Check only the local region whose motifs can change after this edit."""
    flank = max(len(site) for site in TYPEIIS_FORBIDDEN_SITES) - 1
    local_start = max(0, codon_start - flank)
    local_end = min(len(sequence), codon_start + 3 + flank)
    local = _mutated_slice(sequence, local_start, local_end, codon_start, new_codon)
    return not any(site in local for site in TYPEIIS_FORBIDDEN_SITES)


def build_cut_option_index(sequence, allowed_overhangs):
    """
    Index every usable cut once.

    Each option is (overhang, kind, codon_start, new_codon). Native options have
    codon_start/new_codon set to None. Synonymous options store only a tiny patch;
    they do not copy or rescan the complete gene.
    """
    sequence = str(sequence).strip().upper()
    allowed = set(allowed_overhangs)
    if contains_forbidden_typeiis_site(sequence):
        return {}, [], "Input sequence already contains a forbidden Type IIS site"

    by_cut = {}
    for cut in range(4, len(sequence) + 1):
        options = []
        native_window = sequence[cut - 4:cut]
        if native_window in allowed:
            options.append((native_window, "Native", None, None))

        codon_starts = {
            position - (position % 3)
            for position in range(cut - 4, cut)
        }
        for codon_start in codon_starts:
            old_codon = sequence[codon_start:codon_start + 3]
            aa = codon_table.forward_table.get(old_codon)
            if aa is None:
                continue
            for new_codon in syn_map.get(aa, []):
                if new_codon == old_codon:
                    continue
                mutated_window = _mutated_slice(
                    sequence, cut - 4, cut, codon_start, new_codon
                )
                if (
                    mutated_window in allowed
                    and _mutation_is_typeiis_safe(sequence, codon_start, new_codon)
                ):
                    options.append((
                        mutated_window, "Synonymous", codon_start, new_codon
                    ))

        if options:
            unique = []
            seen = set()
            for option in options:
                key = (option[0], option[2], option[3])
                if key not in seen:
                    seen.add(key)
                    unique.append(option)
            by_cut[cut] = unique

    return by_cut, sorted(by_cut), None


def precompute_gene_info(records):
    """Build each gene's cut index once, independent of fragment count or pool."""
    info = {}
    started = time.perf_counter()
    for number, record in enumerate(records, start=1):
        sequence = str(record.seq)
        by_cut, cuts, error = build_cut_option_index(sequence, INTERNAL_OVERHANGS_LIST)
        info[record.id] = {
            "seq": sequence,
            "length": len(sequence),
            "by_cut": by_cut,
            "cuts": cuts,
            "option_count": sum(len(options) for options in by_cut.values()),
            "error": error,
        }
        if SHOW_POOL_PROGRESS:
            print(
                f"[INDEX {number}/{len(records)}] {record.id}: {len(sequence)} nt, "
                f"{len(cuts)} cut positions, {info[record.id]['option_count']} options"
            )
    elapsed = time.perf_counter() - started
    print(f"[TIMING] Indexed {len(records)} multi-fragment genes in {elapsed:.3f} s")
    return info


# ----------------------------
# Fragment-length model
# ----------------------------
def frag_max_len(fragment_index_1based, n_fragments, vec1_penalty_nt):
    if fragment_index_1based == n_fragments:
        return OPOOL_LENGTH - 62
    result = OPOOL_LENGTH - 58
    if fragment_index_1based == 1:
        result -= vec1_penalty_nt
    return result


def max_bases_for_n(n_fragments, vec1_penalty_nt=0):
    """Maximum unique sequence length covered by n overlapping fragments."""
    return (
        sum(
            frag_max_len(index, n_fragments, vec1_penalty_nt)
            for index in range(1, n_fragments + 1)
        )
        - 4 * (n_fragments - 1)
    )


def _remaining_capacity(first_fragment_index, n_fragments, vec1_penalty_nt):
    remaining = list(range(first_fragment_index, n_fragments + 1))
    if not remaining:
        return 0
    return (
        sum(
            frag_max_len(index, n_fragments, vec1_penalty_nt)
            for index in remaining
        )
        - 4 * (len(remaining) - 1)
    )


def minimum_fragment_count(sequence_length, vec1_penalty_nt=0):
    for n_fragments in range(1, MAX_FRAGMENTS + 1):
        if sequence_length <= max_bases_for_n(n_fragments, vec1_penalty_nt):
            return n_fragments
    return None


# ----------------------------
# Dynamic cut-path search (replaces factorial overhang permutations)
# ----------------------------
def _sequence_with_patches(sequence, patches):
    chars = list(sequence)
    for codon_start, new_codon in patches:
        chars[codon_start:codon_start + 3] = new_codon
    return "".join(chars)


def _window_with_patches(sequence, start, end, patches):
    chars = list(sequence[start:end])
    for codon_start, new_codon in patches:
        for position in range(max(start, codon_start), min(end, codon_start + 3)):
            chars[position - start] = new_codon[position - codon_start]
    return "".join(chars)


def find_fragment_path(info, free_overhangs, n_fragments, vec1_penalty_nt):
    """
    Find a valid ordered cut path by searching only feasible cut intervals.

    State tracks the fragment number, next fragment start, overhang bit mask, and
    small synonymous codon patches. Remaining-capacity bounds prune impossible
    branches before any overhang choice is explored.
    """
    sequence = info["seq"]
    sequence_length = info["length"]
    by_cut = info["by_cut"]
    all_cuts = info["cuts"]
    free_mask = 0
    for overhang in free_overhangs:
        free_mask |= OVERHANG_BITS[overhang]

    failed_states = set()
    visited_nodes = 0

    def search(fragment_index, fragment_start, used_mask, patches):
        nonlocal visited_nodes
        visited_nodes += 1

        if fragment_index == n_fragments:
            if sequence_length - fragment_start > frag_max_len(
                fragment_index, n_fragments, vec1_penalty_nt
            ):
                return None
            modified = _sequence_with_patches(sequence, patches)
            if contains_forbidden_typeiis_site(modified):
                return None
            return [], [], patches

        state = (fragment_index, fragment_start, used_mask, patches)
        if state in failed_states:
            return None

        remaining_capacity = _remaining_capacity(
            fragment_index + 1, n_fragments, vec1_penalty_nt
        )
        minimum_cut = max(
            4 if fragment_index == 1 else fragment_start + 5,
            sequence_length + 4 - remaining_capacity,
        )
        maximum_cut = min(
            sequence_length,
            fragment_start + frag_max_len(
                fragment_index, n_fragments, vec1_penalty_nt
            ),
        )
        left = bisect.bisect_left(all_cuts, minimum_cut)
        right = bisect.bisect_right(all_cuts, maximum_cut)
        feasible_cuts = all_cuts[left:right]

        fragments_remaining = n_fragments - fragment_index + 1
        target_fragment_length = math.ceil(
            (
                sequence_length - fragment_start
                + 4 * (fragments_remaining - 1)
            )
            / fragments_remaining
        )
        target_cut = fragment_start + target_fragment_length
        feasible_cuts = sorted(
            feasible_cuts,
            key=lambda cut: (abs(cut - target_cut), -cut),
        )

        current_patches = dict(patches)
        for cut in feasible_cuts:
            options = sorted(
                by_cut[cut],
                key=lambda option: (
                    option[1] != "Native",
                    OVERHANG_RANK[option[0]],
                ),
            )
            for overhang, _kind, codon_start, new_codon in options:
                bit = OVERHANG_BITS[overhang]
                if not (free_mask & bit) or (used_mask & bit):
                    continue

                next_patches = dict(current_patches)
                if codon_start is not None:
                    if (
                        codon_start in next_patches
                        and next_patches[codon_start] != new_codon
                    ):
                        continue
                    next_patches[codon_start] = new_codon
                next_patches_tuple = tuple(sorted(next_patches.items()))

                if _window_with_patches(
                    sequence, cut - 4, cut, next_patches_tuple
                ) != overhang:
                    continue

                result = search(
                    fragment_index + 1,
                    cut - 4,
                    used_mask | bit,
                    next_patches_tuple,
                )
                if result is not None:
                    later_cuts, later_overhangs, final_patches = result
                    return (
                        [cut, *later_cuts],
                        [overhang, *later_overhangs],
                        final_patches,
                    )

        failed_states.add(state)
        return None

    result = search(1, 0, 0, tuple())
    if result is None:
        return None, visited_nodes

    cuts, overhangs, patches = result
    modified_sequence = _sequence_with_patches(sequence, patches)
    return {
        "cuts": cuts,
        "overhangs": overhangs,
        "patches": patches,
        "sequence": modified_sequence,
    }, visited_nodes


# ----------------------------
# Pool construction
# ----------------------------
def choose_vec1_from_free(free_vec1_set):
    free = {value.upper() for value in free_vec1_set}
    w2 = sorted(free & VEC1_W2_SET)
    if w2:
        return w2[0], "W2", 0
    w0 = sorted(free & VEC1_W0_SET)
    if w0:
        return w0[0], "W0", VEC1_WINDOW0_FRAG1_PENALTY_NT
    return None, None, None


def effective_long_pool_capacity(n_fragments):
    internal_capacity = len(INTERNAL_OVERHANGS_LIST) // (n_fragments - 1)
    capacities = [internal_capacity]
    if VECTOR_OH1_MODE == "ser_windows":
        capacities.append(len(VEC1_SER_UNION))
    if GENES_PER_SUBPOOL is not None:
        capacities.append(GENES_PER_SUBPOOL)
    return min(capacities)


def _fragments_from_cuts(sequence, cuts):
    fragments = []
    start = 0
    for cut in cuts:
        fragments.append(sequence[start:cut])
        start = cut - 4
    fragments.append(sequence[start:])
    return fragments


def build_fast_pools_for_n(genes, gene_info, n_fragments, start_block_index):
    """Greedily pack genes while dynamic search adapts to each pool's free overhangs."""
    rows = []
    remaining = sorted(
        genes,
        key=lambda record: (
            gene_info[record.id]["option_count"],
            -gene_info[record.id]["length"],
            record.id,
        ),
    )
    next_block = start_block_index
    pool_capacity = effective_long_pool_capacity(n_fragments)
    total_search_nodes = 0

    if pool_capacity <= 0:
        return rows, remaining, next_block, total_search_nodes

    while remaining:
        pool_records = []
        free_internal = set(INTERNAL_OVERHANGS_LIST)
        free_vec1 = set(VEC1_SER_UNION) if VECTOR_OH1_MODE == "ser_windows" else set()

        index = 0
        while index < len(remaining) and len(pool_records) < pool_capacity:
            record = remaining[index]
            info = gene_info[record.id]
            if info["error"] or not info["cuts"]:
                index += 1
                continue

            if VECTOR_OH1_MODE == "fixed":
                vec1, vec1_source, penalty = VECTOR_OH1_FIXED, "Fixed", 0
            else:
                vec1, vec1_source, penalty = choose_vec1_from_free(free_vec1)
                if vec1 is None:
                    break

            if info["length"] > max_bases_for_n(n_fragments, penalty):
                index += 1
                continue

            result, visited_nodes = find_fragment_path(
                info, free_internal, n_fragments, penalty
            )
            total_search_nodes += visited_nodes
            if result is None:
                index += 1
                continue

            pool_records.append((record, vec1, vec1_source, penalty, result))
            free_internal.difference_update(result["overhangs"])
            if VECTOR_OH1_MODE == "ser_windows":
                free_vec1.remove(vec1)
            remaining.pop(index)

        if not pool_records:
            break

        for record, vec1, vec1_source, penalty, result in pool_records:
            full_sequence = result["sequence"]
            fragments = _fragments_from_cuts(full_sequence, result["cuts"])
            row = {
                "Block": next_block,
                "Length Distribution": f"Long-{n_fragments}part",
                "Sequence Name": record.id,
                "VectorOH1": vec1,
                "VectorOH1_Source": vec1_source,
                "VectorOH2": VECTOR_OH2,
                "Full Sequence": full_sequence,
            }
            for number, overhang in enumerate(result["overhangs"], start=1):
                row[f"Overhang{number}"] = overhang
            for number, fragment in enumerate(fragments, start=1):
                row[f"DNA Fragment {number}"] = fragment
            rows.append(row)

        if SHOW_POOL_PROGRESS:
            print(
                f"[BLOCK {next_block}] {n_fragments} fragments: "
                f"assigned {len(pool_records)} gene(s); "
                f"{len(remaining)} eligible gene(s) remain"
            )
        next_block += 1

    return rows, remaining, next_block, total_search_nodes


# ----------------------------
# Single-fragment pools
# ----------------------------
def effective_short_pool_capacity(number_of_short_genes):
    capacities = [max(1, number_of_short_genes)]
    if SHORT_POOL_MAX_SIZE is not None:
        capacities.append(SHORT_POOL_MAX_SIZE)
    if GENES_PER_SUBPOOL is not None:
        capacities.append(GENES_PER_SUBPOOL)
    if VECTOR_OH1_MODE == "ser_windows":
        capacities.append(len(VEC1_SER_UNION))
    return min(capacities)


def _ordered_ser_vec1_options():
    return [
        *((value, "W2") for value in sorted(VEC1_W2_SET)),
        *((value, "W0") for value in sorted(VEC1_W0_SET)),
    ]


def assign_short_phase(short_records, start_block):
    if not short_records:
        return []
    rows = []
    block_size = effective_short_pool_capacity(len(short_records))
    for chunk_start in range(0, len(short_records), block_size):
        block = start_block + (chunk_start // block_size)
        chunk = short_records[chunk_start:chunk_start + block_size]
        vec1_options = iter(_ordered_ser_vec1_options())
        for record in chunk:
            if VECTOR_OH1_MODE == "fixed":
                vec1, vec1_source = VECTOR_OH1_FIXED, "Fixed"
            else:
                vec1, vec1_source = next(vec1_options)
            rows.append({
                "Block": block,
                "Length Distribution": "Short",
                "Sequence Name": record.id,
                "VectorOH1": vec1,
                "VectorOH1_Source": vec1_source,
                "VectorOH2": VECTOR_OH2,
                "Overhang1": "N/A",
                "Overhang2": "N/A",
                "DNA Fragment 1": str(record.seq),
                "DNA Fragment 2": "",
                "DNA Fragment 3": "",
                "Full Sequence": str(record.seq),
            })
    return rows


# ----------------------------
# Output validation
# ----------------------------
def validate_pool_assignment(df_out, source_records):
    """Fail before writing if any core assembly invariant is broken."""
    if df_out.empty:
        return
    source_sequences = {record.id: str(record.seq) for record in source_records}
    fragment_columns = sorted(
        [column for column in df_out.columns if column.startswith("DNA Fragment ")],
        key=lambda column: int(column.replace("DNA Fragment ", "")),
    )
    overhang_columns = sorted(
        [column for column in df_out.columns if column.startswith("Overhang")],
        key=lambda column: int(column.replace("Overhang", "")),
    )

    for _, row in df_out.iterrows():
        sequence_name = str(row["Sequence Name"])
        full_sequence = str(row["Full Sequence"])
        fragments = [
            str(row[column]).strip().upper()
            for column in fragment_columns
            if pd.notna(row.get(column)) and str(row.get(column)).strip() not in {"", "nan"}
        ]
        overhangs = [
            str(row[column]).strip().upper()
            for column in overhang_columns
            if pd.notna(row.get(column)) and str(row.get(column)).strip().upper() not in {"", "N/A", "NAN"}
        ]
        if len(overhangs) != max(0, len(fragments) - 1):
            raise AssertionError(f"{sequence_name}: fragment/overhang count mismatch")
        reconstructed = fragments[0] + "".join(fragment[4:] for fragment in fragments[1:])
        if reconstructed != full_sequence:
            raise AssertionError(f"{sequence_name}: fragments do not reconstruct Full Sequence")
        for index, overhang in enumerate(overhangs):
            if fragments[index][-4:] != overhang or fragments[index + 1][:4] != overhang:
                raise AssertionError(f"{sequence_name}: incorrect overlap at cut {index + 1}")
        if len(overhangs) != len(set(overhangs)):
            raise AssertionError(f"{sequence_name}: an internal overhang is reused")
        penalty = (
            VEC1_WINDOW0_FRAG1_PENALTY_NT
            if str(row["VectorOH1_Source"]).upper() == "W0"
            else 0
        )
        for index, fragment in enumerate(fragments, start=1):
            maximum = frag_max_len(index, len(fragments), penalty)
            if len(fragment) > maximum:
                raise AssertionError(
                    f"{sequence_name}: fragment {index} is {len(fragment)} nt; max is {maximum}"
                )
        if contains_forbidden_typeiis_site(full_sequence):
            raise AssertionError(f"{sequence_name}: Full Sequence contains a forbidden Type IIS site")
        source_translation = str(Seq(source_sequences[sequence_name]).translate())
        output_translation = str(Seq(full_sequence).translate())
        if output_translation != source_translation:
            raise AssertionError(f"{sequence_name}: synonymous edits changed translation")

    for block, block_df in df_out.groupby("Block"):
        internal = []
        for column in overhang_columns:
            internal.extend(
                str(value).strip().upper()
                for value in block_df[column]
                if pd.notna(value) and str(value).strip().upper() not in {"", "N/A", "NAN"}
            )
        if len(internal) != len(set(internal)):
            raise AssertionError(f"Block {block}: an internal overhang is reused within the sub-pool")
        if GENES_PER_SUBPOOL is not None and len(block_df) > GENES_PER_SUBPOOL:
            raise AssertionError(
                f"Block {block}: contains {len(block_df)} genes; configured maximum is {GENES_PER_SUBPOOL}"
            )
        if VECTOR_OH1_MODE == "ser_windows":
            vec1_values = list(block_df["VectorOH1"])
            if len(vec1_values) != len(set(vec1_values)):
                raise AssertionError(f"Block {block}: VectorOH1 is reused")

    print(f"[VALIDATION] Passed all checks for {len(df_out)} assigned genes")


# ----------------------------
# Main pipeline
# ----------------------------
def validate_pool_settings():
    if not isinstance(OPOOL_LENGTH, int) or OPOOL_LENGTH <= 62:
        raise ValueError("OPOOL_LENGTH must be an integer greater than 62.")
    if VECTOR_OH1_MODE not in {"fixed", "ser_windows"}:
        raise ValueError("VECTOR_OH1_MODE must be 'fixed' or 'ser_windows'.")
    if GENES_PER_SUBPOOL is not None and (
        not isinstance(GENES_PER_SUBPOOL, int) or GENES_PER_SUBPOOL < 1
    ):
        raise ValueError("GENES_PER_SUBPOOL must be None or a positive integer.")
    if SHORT_POOL_MAX_SIZE is not None and (
        not isinstance(SHORT_POOL_MAX_SIZE, int) or SHORT_POOL_MAX_SIZE < 1
    ):
        raise ValueError("SHORT_POOL_MAX_SIZE must be None or a positive integer.")
    if not isinstance(MAX_FRAGMENTS, int) or MAX_FRAGMENTS < 2:
        raise ValueError("MAX_FRAGMENTS must be an integer of at least 2.")
    if VECTOR_OH2 in set(OVERHANGS_LIST):
        raise ValueError("Overhang file contains VECTOR_OVERHANG_2; remove it.")
    if VECTOR_OH1_MODE == "fixed" and VECTOR_OH1_FIXED in set(OVERHANGS_LIST):
        raise ValueError("Overhang file contains VECTOR_OVERHANG_1_FIXED; remove it.")


def main_pipeline():
    validate_pool_settings()
    pipeline_started = time.perf_counter()
    records, strip_log = load_sequences(INPUT_FILE)

    if STRIP_NTERM_MET:
        print(f"[INFO] Removed leading ATG from {len(strip_log)} sequence(s).")
        if WRITE_STRIP_LOG:
            Path(STRIP_LOG_CSV).parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(strip_log).to_csv(STRIP_LOG_CSV, index=False)
            print(f"[INFO] Wrote strip log -> {STRIP_LOG_CSV}")

    short_limit = OPOOL_LENGTH - 62
    short_records = [record for record in records if len(record.seq) <= short_limit]
    pending = [record for record in records if len(record.seq) > short_limit]

    rows = assign_short_phase(short_records, start_block=1)
    current_block = max((row["Block"] for row in rows), default=0) + 1
    gene_info = precompute_gene_info(pending)
    total_search_nodes = 0

    for n_fragments in range(2, MAX_FRAGMENTS + 1):
        if not pending:
            break
        if n_fragments - 1 > len(INTERNAL_OVERHANGS_LIST):
            break

        eligible = [
            record for record in pending
            if len(record.seq) <= max_bases_for_n(n_fragments, 0)
        ]
        if not eligible:
            continue

        if SHOW_POOL_PROGRESS:
            print(
                f"[SEARCH] Trying {len(eligible)} eligible gene(s) with "
                f"{n_fragments} fragments; pool capacity "
                f"{effective_long_pool_capacity(n_fragments)}"
            )
        new_rows, failed, next_block, visited_nodes = build_fast_pools_for_n(
            eligible, gene_info, n_fragments, current_block
        )
        total_search_nodes += visited_nodes
        rows.extend(new_rows)
        current_block = next_block
        assigned_ids = {str(row["Sequence Name"]) for row in new_rows}
        pending = [record for record in pending if record.id not in assigned_ids]

    unassigned = [record.id for record in pending]
    df_out = pd.DataFrame(rows)
    if not df_out.empty:
        df_out.sort_values(by=["Block", "Sequence Name"], inplace=True)
        overhang_columns = sorted(
            [column for column in df_out.columns if column.startswith("Overhang")],
            key=lambda column: int(column.replace("Overhang", "")),
        )
        fragment_columns = sorted(
            [column for column in df_out.columns if column.startswith("DNA Fragment ")],
            key=lambda column: int(column.replace("DNA Fragment ", "")),
        )
        base_columns = ["Block", "Length Distribution", "Sequence Name"]
        vector_columns = ["VectorOH1", "VectorOH1_Source", "VectorOH2"]
        final_order = [
            *base_columns,
            *vector_columns,
            *overhang_columns,
            *fragment_columns,
            "Full Sequence",
        ]
        df_out = df_out[[column for column in final_order if column in df_out.columns]]

    validate_pool_assignment(df_out, records)
    df_out.to_csv(OUTPUT_FILE, index=False)
    write_unassigned_log(unassigned, UNASSIGNED_LOG)
    used_overhangs = write_used_overhangs(df_out, OVERHANGS_USED_CSV)

    elapsed = time.perf_counter() - pipeline_started
    print("\nProcessing complete.")
    print(f"OPOOL_LENGTH          = {OPOOL_LENGTH}")
    print(f"GENES_PER_SUBPOOL     = {GENES_PER_SUBPOOL}")
    print(f"VECTOR_OH1_MODE       = {VECTOR_OH1_MODE}")
    print(f"VectorOH2             = {VECTOR_OH2}")
    print(f"Internal OH count     = {len(INTERNAL_OVERHANGS_LIST)}")
    print(f"Search states visited = {total_search_nodes:,}")
    print(f"Used overhangs        = {OVERHANGS_USED_CSV} ({len(used_overhangs)} unique)")
    print(f"Final CSV             = {OUTPUT_FILE}")
    print(f"Total runtime         = {elapsed:.3f} seconds")
    if unassigned:
        print(f"Unassigned            = {UNASSIGNED_LOG} ({len(unassigned)} sequences)")
    else:
        print("All genes assigned.")
    return df_out


if __name__ == "__main__":
    pool_assignment_df = main_pipeline()


## Primer assignment
Contains ability to use orthogonal oligos as unique primer pairs or combinatorially. Also handles compatibility with different vector overhang sources, for instance adding 2nt to finish of the 2 serine codons if using the approach for cell-free assembly

### Primer assignment with stuffer

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio.Seq import Seq

# =========================
# USER CONFIG / VARIABLES
# =========================

# Target oligo length (final: FWD + stuffer + (TypeIIS recognition+N) + insert(+vecOHs) + rc(recognition+N) + rc(REV))
OPOOL_LENGTH = 350  # e.g. 300/250/200/150/100

# Optional stuffer to normalize all oligos to OPOOL_LENGTH
ADD_STUFFER = True
STUFFER_GC_MIN = 0.40
STUFFER_GC_MAX = 0.60
STUFFER_MAX_HOMOPOLYMER = 4
STUFFER_MAX_TRIES = 10000
STUFFER_SEED = 123  # set None for non-deterministic

# Type IIS element: the recognition site is set once in Gene optimisation variables.
# Keep its required N base separate; only the recognition site is used for motif exclusion.
TYPEIIS_N_SEQUENCE = "A"  # exactly one A/C/G/T base appended to the recognition site

# Primer assignment mode
#   - "unique_pairs": PRIMER_CSV is [F,R,F,R,...] and assigns one sequential pair per Block
#   - "combinatorial": PAIRS_CSV lists (Fwd,Rev) pairs; assigns one row per Block
PRIMER_ASSIGN_MODE = "combinatorial"   # "unique_pairs" or "combinatorial"

# Unique-pairs mode settings
PRIMER_START_AT = None   # None/"" -> first primer; else match first column (numeric or string)
BLOCK_BASE      = 1

if "RUN_PATHS" not in globals():
    raise RuntimeError("Run the Gene optimisation variables cell first so RUN_PATHS is defined.")

# Primer inventory
PRIMER_CSV            = RUN_PATHS["primers"]
OUTPUT_UNUSED_PRIMERS = RUN_PATHS["unused_primers"]
WRITE_UNUSED_PRIMERS  = True

# Combinatorial mode settings
GENERATE_PAIR_TABLE_ONLY = False
ALL_PAIRS_CSV            = RUN_PATHS["all_pairs"]
PAIRS_CSV                = RUN_PATHS["unused_pairs"] #all_pairs
OUTPUT_UNUSED_PAIRS      = RUN_PATHS["unused_pairs"]

# Assembly + outputs
INPUT_ASSEMBLY_CSV    = RUN_PATHS["assigned"]
OUTPUT_WITH_PRIMERS   = RUN_PATHS["full_info"]
OUTPUT_FASTA          = RUN_PATHS["references_fasta"]
OUTPUT_FRAGMENTS_CSV  = RUN_PATHS["fragments"]

MAKE_OUTPUT_DIRS = True

# Vector overhang handling
# Assigned CSV may contain VectorOH1/VectorOH2/VectorOH1_Source; if missing or N/A we fall back.
FALLBACK_VECTOR_OH1 = "TATG"
FALLBACK_VECTOR_OH2 = "GGAT"

# Which VectorOH1_Source labels require +2nt completion (your W0/window0 case)
VEC1_NEEDS_2NT_SOURCE_LABELS = {"W0"}


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations, product
from pathlib import Path
from Bio.Seq import Seq

# =========================
# RNG
# =========================
_rng = np.random.default_rng(STUFFER_SEED) if (STUFFER_SEED is not None) else np.random.default_rng()

# =========================
# HELPERS
# =========================

def revcomp(s: str) -> str:
    return str(Seq(s).reverse_complement())

def validate_dna(value: str, label: str, exact_length=None) -> str:
    """Normalize an A/C/G/T setting and fail early on invalid configuration."""
    sequence = str(value).strip().upper()
    if not sequence or any(base not in "ACGT" for base in sequence):
        raise ValueError(f"{label} must contain only A/C/G/T and cannot be empty.")
    if exact_length is not None and len(sequence) != exact_length:
        raise ValueError(f"{label} must be exactly {exact_length} nucleotide(s).")
    return sequence

def typeiis_forbidden_sites(recognition_site: str) -> list:
    """Return the recognition sequence and its reverse complement only."""
    recognition_site = validate_dna(recognition_site, "TYPEIIS_RECOGNITION_SITE")
    return sorted({recognition_site, revcomp(recognition_site)})

def same_order_subsequences(sequence: str, subsequence_length: int) -> set:
    """Return subsequences whose bases retain their order in the source sequence."""
    sequence = str(sequence).strip().upper()
    if subsequence_length <= 0 or len(sequence) < subsequence_length:
        return set()
    return {
        "".join(sequence[i] for i in indices)
        for indices in combinations(range(len(sequence)), subsequence_length)
    }

def typeiis_forbidden_stuffer_prefixes(recognition_site: str, prefix_length: int = 5) -> set:
    """Return same-order prefixes forbidden at the first bases of the stuffer."""
    forbidden_sites = typeiis_forbidden_sites(recognition_site)
    return set().union(*(
        same_order_subsequences(site, prefix_length)
        for site in forbidden_sites
    ))

def is_nonempty_seq(x) -> bool:
    """True if x is a non-empty string (not NaN, not N/A)."""
    if x is None:
        return False
    if isinstance(x, float) and np.isnan(x):
        return False
    s = str(x).strip()
    return s != "" and s.upper() != "N/A"

def load_primers(path: str):
    df = pd.read_csv(path, header=None)
    df = df.replace(r'^\s*$', np.nan, regex=True).dropna(how='all').reset_index(drop=True)
    if df.shape[1] < 2:
        raise ValueError("Primer CSV must have ≥2 columns: first=Name/ID, last=Sequence.")
    name_col, seq_col = 0, df.shape[1] - 1
    names = df.iloc[:, name_col].astype(str).str.strip()
    seqs  = df.iloc[:, seq_col].astype(str).str.strip().str.upper()
    pairs = [(n, s) for n, s in zip(names, seqs) if n and s]
    if not pairs:
        raise ValueError("No valid (name, seq) rows parsed from primer CSV.")
    return pairs

def names_equal(a, b) -> bool:
    sa = str(a).strip(); sb = str(b).strip()
    try:
        return int(sa) == int(sb)
    except Exception:
        return sa == sb

def generate_all_pairs(primers, out_csv: str):
    rows = []
    n = len(primers)
    pair_id = 1
    for i in range(n):
        f_name, f_seq = primers[i]
        for j in range(i + 1, n):
            r_name, r_seq = primers[j]
            rows.append({
                "Pair_ID": pair_id,
                "Fwd_Name": f_name,
                "Fwd_Seq":  f_seq,
                "Rev_Name": r_name,
                "Rev_Seq":  r_seq,
            })
            pair_id += 1
    df_pairs = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df_pairs.to_csv(out_csv, index=False)
    print(f"[INFO] Generated {len(df_pairs)} unique primer pairs from {n} primers -> {out_csv}")

def ensure_pair_table(primers, pairs_csv, all_pairs_csv):
    pairs_path = Path(pairs_csv)
    if pairs_path.exists():
        return
    generate_all_pairs(primers, all_pairs_csv)
    pairs_path.parent.mkdir(parents=True, exist_ok=True)
    pd.read_csv(all_pairs_csv).to_csv(pairs_path, index=False)
    print(f"[INFO] Initialized primer pair table -> {pairs_path}")

def write_unused_primers(primers, assigned_df, output_csv):
    """Write unused individual oligos after either primer-assignment mode."""
    used_names = (
        set(assigned_df["Primer_Forward_Name"].astype(str).str.strip())
        | set(assigned_df["Primer_Reverse_Name"].astype(str).str.strip())
    )
    unused_primers = [
        (name, seq) for name, seq in primers
        if str(name).strip() not in used_names
    ]
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(unused_primers, columns=["Primer_Name", "Sequence"]).to_csv(
        output_csv, index=False, header=False
    )
    print(
        f"[INFO] Wrote unused primers -> {output_csv} "
        f"({len(unused_primers)} of {len(primers)} remain; no header)."
    )

# =========================
# Ser-Ser completion logic (W0/window0 adds +2 nt)
# =========================
SER_CODONS = ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"]

def build_vec1_w0_suffix_map():
    """
    For SerSer = c1+c2 (6 nt), window0 4-mer is bases[0:4].
    If that 4-mer is used as VectorOH1, we need to add bases[4:6] (2 nt).
    Returns dict: vec1_4mer -> sorted list of possible 2nt suffixes.
    """
    m = {}
    for c1 in SER_CODONS:
        for c2 in SER_CODONS:
            s = c1 + c2
            vec1 = s[0:4]
            suf2 = s[4:6]
            m.setdefault(vec1, set()).add(suf2)
    return {k: sorted(list(v)) for k, v in m.items()}

VEC1_W0_SUFFIXES = build_vec1_w0_suffix_map()

def normalize_source_label(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip().upper()

def vec_overhangs_for_row(row):
    """
    Always returns usable (vec1, vec2, vec1_src):
      - if VectorOH1/2 are missing or N/A -> fall back
    """
    vec1 = row.get("VectorOH1", "")
    vec2 = row.get("VectorOH2", "")
    vec1_src = row.get("VectorOH1_Source", "")

    vec1 = str(vec1).strip().upper() if is_nonempty_seq(vec1) else ""
    vec2 = str(vec2).strip().upper() if is_nonempty_seq(vec2) else ""
    vec1_src = normalize_source_label(vec1_src)

    if not vec1:
        vec1 = FALLBACK_VECTOR_OH1.strip().upper()
    if not vec2:
        vec2 = FALLBACK_VECTOR_OH2.strip().upper()

    return vec1, vec2, vec1_src

def vec1_prefix_with_optional_2nt(vec1_4mer: str, vec1_source_label: str) -> str:
    """
    If vec1_source_label indicates W0/window0 case, return vec1 + extra2.
    Otherwise return vec1.
    """
    if vec1_source_label in {s.upper() for s in VEC1_NEEDS_2NT_SOURCE_LABELS}:
        opts = VEC1_W0_SUFFIXES.get(vec1_4mer, None)
        if not opts:
            raise ValueError(
                f"VectorOH1_Source='{vec1_source_label}' requires +2nt completion, "
                f"but vec1='{vec1_4mer}' is not a valid SerSer window0 4-mer."
            )
        extra2 = opts[0]  # deterministic
        return vec1_4mer + extra2
    return vec1_4mer

# =========================
# STUFFER GENERATION (ACGT only)
# =========================

def gc_fraction(seq: str) -> float:
    seq = seq.upper()
    gc = sum(1 for b in seq if b in ("G", "C"))
    return gc / len(seq) if seq else 0.0

def has_homopolymer_over(seq: str, max_run: int) -> bool:
    if not seq:
        return False
    run = 1
    prev = seq[0]
    for b in seq[1:]:
        if b == prev:
            run += 1
            if run > max_run:
                return True
        else:
            prev = b
            run = 1
    return False

def creates_forbidden_site_across_junction(left: str, right: str, forbidden_sites: list) -> bool:
    """Return True only when a forbidden site spans the left/right boundary."""
    forbidden_sites = [site.upper() for site in forbidden_sites if site]
    if not forbidden_sites:
        return False
    flank_length = max(len(site) for site in forbidden_sites) - 1
    left_tail = str(left).strip().upper()[-flank_length:] if flank_length else ""
    right_head = str(right).strip().upper()[:flank_length] if flank_length else ""
    combined = left_tail + right_head
    boundary = len(left_tail)
    for site in forbidden_sites:
        start = combined.find(site)
        while start != -1:
            if start < boundary and start + len(site) > boundary:
                return True
            start = combined.find(site, start + 1)
    return False

def stuffer_candidate_is_valid(
    sequence: str,
    forbidden_sites: list,
    forbidden_prefixes: set,
    left_context: str,
    right_context: str,
    prefix_length: int = 5,
    enforce_gc: bool = True,
) -> bool:
    """Apply all stuffer-internal, prefix, and junction constraints."""
    sequence = sequence.upper()
    if any(base not in "ACGT" for base in sequence):
        return False
    if enforce_gc and not (STUFFER_GC_MIN <= gc_fraction(sequence) <= STUFFER_GC_MAX):
        return False
    if has_homopolymer_over(sequence, STUFFER_MAX_HOMOPOLYMER):
        return False
    if any(site in sequence for site in forbidden_sites):
        return False
    if len(sequence) >= prefix_length and sequence[:prefix_length] in forbidden_prefixes:
        return False
    if creates_forbidden_site_across_junction(left_context, sequence, forbidden_sites):
        return False
    if creates_forbidden_site_across_junction(sequence, right_context, forbidden_sites):
        return False
    return True

def make_stuffer(
    length: int,
    forbidden_sites: list,
    forbidden_prefixes: set,
    left_context: str = "",
    right_context: str = "",
    prefix_length: int = 5,
) -> str:
    """
    Random A/C/G/T stuffer of exact length that:
      - has GC in [min,max] when feasible
      - has no homopolymer run > max
      - contains neither the Type IIS recognition site nor its reverse complement
      - does not begin with a forbidden same-order 5-base recognition subsequence
      - cannot create either recognition orientation across either stuffer junction
      - NEVER contains N
    """
    if length < 0:
        raise ValueError("Stuffer length cannot be negative.")

    forbidden_sites = [site.upper() for site in forbidden_sites if site]
    forbidden_prefixes = {prefix.upper() for prefix in forbidden_prefixes if prefix}

    if length == 0:
        if stuffer_candidate_is_valid(
            "", forbidden_sites, forbidden_prefixes, left_context, right_context,
            prefix_length=prefix_length, enforce_gc=False,
        ):
            return ""
        raise RuntimeError("A zero-length stuffer would create a forbidden Type IIS junction.")

    # Exhaustively search tiny stuffers; use the closest feasible GC fraction to the target band.
    if length < 5:
        candidates = ["".join(chars) for chars in product("ACGT", repeat=length)]
        _rng.shuffle(candidates)

        def gc_band_distance(candidate):
            gcf = gc_fraction(candidate)
            return max(STUFFER_GC_MIN - gcf, 0.0, gcf - STUFFER_GC_MAX)

        best_gc_distance = min(gc_band_distance(candidate) for candidate in candidates)
        for candidate in candidates:
            if gc_band_distance(candidate) != best_gc_distance:
                continue
            if stuffer_candidate_is_valid(
                candidate, forbidden_sites, forbidden_prefixes, left_context, right_context,
                prefix_length=prefix_length, enforce_gc=False,
            ):
                return candidate
        raise RuntimeError(f"No valid stuffer exists for length {length} with the current junctions.")

    bases = np.array(list("ACGT"))
    for _ in range(STUFFER_MAX_TRIES):
        candidate = "".join(_rng.choice(bases, size=length, replace=True))
        if stuffer_candidate_is_valid(
            candidate, forbidden_sites, forbidden_prefixes, left_context, right_context,
            prefix_length=prefix_length, enforce_gc=True,
        ):
            return candidate

    raise RuntimeError(
        f"Failed to generate valid stuffer of length {length} after {STUFFER_MAX_TRIES} tries."
    )

# =========================
# CORE ASSEMBLY
# =========================

def apply_pair_to_block_rows(df_block: pd.DataFrame,
                             fwd_name: str, fwd_seq: str,
                             rev_name: str, rev_seq: str,
                             frag_cols: list) -> pd.DataFrame:
    """
    For each row:
      - determine vec1/vec2 (row-specific; fallback if N/A/missing)
      - add vec1(+optional2nt) to FIRST non-empty fragment
      - add vec2 to LAST non-empty fragment (even if only one fragment -> both applied)
      - build oligo: FWD + stuffer + (TypeIIS recognition+N) + insert + rc(recognition+N) + rc(REV)
      - if ADD_STUFFER: make total length exactly OPOOL_LENGTH
    """
    recognition_site = validate_dna(TYPEIIS_RECOGNITION_SITE, "TYPEIIS_RECOGNITION_SITE")
    n_sequence = validate_dna(TYPEIIS_N_SEQUENCE, "TYPEIIS_N_SEQUENCE", exact_length=1)
    if len(recognition_site) < 5:
        raise ValueError("TYPEIIS_RECOGNITION_SITE must be at least 5 nucleotides long.")

    # Primer-addition elements require recognition+N in the forward orientation
    # and the reverse complement of that complete element in the reverse orientation.
    cut_pref = recognition_site + n_sequence
    cut_suf  = revcomp(cut_pref)
    rev_seq_suf = revcomp(rev_seq)

    # Stuffer rules use only the recognition sequence, never the separate N base.
    forbidden_sites = typeiis_forbidden_sites(recognition_site)
    forbidden_prefixes = typeiis_forbidden_stuffer_prefixes(recognition_site, prefix_length=5)

    df_block = df_block.copy()

    for idx, row in df_block.iterrows():
        vec1, vec2, vec1_src = vec_overhangs_for_row(row)
        vec1_prefix = vec1_prefix_with_optional_2nt(vec1, vec1_src)

        non_empty_idx = [i for i, col in enumerate(frag_cols) if is_nonempty_seq(row.get(col, None))]

        new_frags = []
        for i, col in enumerate(frag_cols):
            seq = row.get(col, None)

            if not is_nonempty_seq(seq):
                # normalize missing to empty string
                new_frags.append("")
                continue

            seq = str(seq).strip().upper()

            first = (i == non_empty_idx[0]) if non_empty_idx else False
            last  = (i == non_empty_idx[-1]) if non_empty_idx else False

            s = seq
            if first:
                s = vec1_prefix + s
            if last:
                s = s + vec2

            core = cut_pref + s + cut_suf

            if ADD_STUFFER:
                fixed_len = len(fwd_seq) + len(core) + len(rev_seq_suf)
                stuffer_len = OPOOL_LENGTH - fixed_len
                if stuffer_len < 0:
                    raise ValueError(
                        f"Oligo would exceed OPOOL_LENGTH for row '{row.get('Sequence Name','?')}', "
                        f"Block={row.get('Block','?')}, fragment='{col}'. "
                        f"Need OPOOL_LENGTH≥{fixed_len} but OPOOL_LENGTH={OPOOL_LENGTH}."
                    )
                stuffer = make_stuffer(
                    stuffer_len,
                    forbidden_sites=forbidden_sites,
                    forbidden_prefixes=forbidden_prefixes,
                    left_context=fwd_seq,
                    right_context=cut_pref,
                    prefix_length=5,
                )
            else:
                stuffer = ""

            oligo = fwd_seq + stuffer + core + rev_seq_suf

            if ADD_STUFFER and len(oligo) != OPOOL_LENGTH:
                raise AssertionError(f"Built oligo length {len(oligo)} != OPOOL_LENGTH {OPOOL_LENGTH}")
            if any(b not in "ACGT" for b in oligo):
                raise AssertionError("Found non-ACGT base in oligo (stuffer must be A/C/G/T only).")

            new_frags.append(oligo)

        for col, nf in zip(frag_cols, new_frags):
            df_block.at[idx, col] = nf

        df_block.at[idx, "Primer_Forward_Name"] = fwd_name
        df_block.at[idx, "Primer_Forward_Seq"]  = fwd_seq
        df_block.at[idx, "Primer_Reverse_Name"] = rev_name
        df_block.at[idx, "Primer_Reverse_Seq"]  = rev_seq

    return df_block

def ensure_output_dirs():
    if not MAKE_OUTPUT_DIRS:
        return
    for p in [OUTPUT_WITH_PRIMERS, OUTPUT_FASTA, OUTPUT_FRAGMENTS_CSV,
              OUTPUT_UNUSED_PRIMERS, ALL_PAIRS_CSV, OUTPUT_UNUSED_PAIRS]:
        if p:
            Path(p).parent.mkdir(parents=True, exist_ok=True)

# =========================
# MAIN
# =========================

ensure_output_dirs()

df = pd.read_csv(INPUT_ASSEMBLY_CSV)

if "Block" not in df.columns:
    raise ValueError("Input assembly CSV must contain a 'Block' column.")
df["Block"] = pd.to_numeric(df["Block"], errors="raise").astype(int)

frag_cols = [c for c in df.columns if c.startswith("DNA Fragment")]
if not frag_cols:
    raise ValueError("No 'DNA Fragment' columns found in INPUT_ASSEMBLY_CSV.")
frag_cols_sorted = sorted(frag_cols, key=lambda x: int(x.replace("DNA Fragment ", "")))

unique_blocks = sorted(df["Block"].unique())
num_blocks = len(unique_blocks)

if "VectorOH1" not in df.columns:
    print("[WARN] 'VectorOH1' column not found in assigned CSV. Falling back to FALLBACK_VECTOR_OH1.")
if "VectorOH2" not in df.columns:
    print("[WARN] 'VectorOH2' column not found in assigned CSV. Falling back to FALLBACK_VECTOR_OH2.")
if "VectorOH1_Source" not in df.columns:
    print("[INFO] 'VectorOH1_Source' not found — will NOT add +2nt SerSer completion unless present.")

primers = load_primers(PRIMER_CSV)

if PRIMER_ASSIGN_MODE == "combinatorial" and GENERATE_PAIR_TABLE_ONLY:
    generate_all_pairs(primers, ALL_PAIRS_CSV)
    print("[DONE] Pair table generation only.")
    raise SystemExit(0)

# =========================
# MODE 1: UNIQUE PAIRS
# =========================
if PRIMER_ASSIGN_MODE == "unique_pairs":
    if PRIMER_START_AT is None or (isinstance(PRIMER_START_AT, str) and PRIMER_START_AT.strip() == ""):
        start_idx = 0
        print(f"[INFO] PRIMER_START_AT not specified — using FIRST primer in {PRIMER_CSV} as forward primer for Block={BLOCK_BASE}.")
    else:
        start_idx = next((i for i, (nm, _) in enumerate(primers) if names_equal(nm, PRIMER_START_AT)), None)
        if start_idx is None:
            raise ValueError(
                f"Primer named/ID '{PRIMER_START_AT}' not found in {PRIMER_CSV} (first column). "
                "Set PRIMER_START_AT=None to start from the first primer."
            )
        print(f"[INFO] Using primer '{primers[start_idx][0]}' as the explicit start primer for Block={BLOCK_BASE}.")

    if start_idx % 2 == 1:
        print(
            f"[WARN] Start primer '{primers[start_idx][0]}' is at odd index {start_idx} (0-based). "
            f"If your file is [F,R,F,R,...], this may be a reverse. Using it as FORWARD as requested."
        )

    def primers_for_block(block_val: int):
        k = block_val - BLOCK_BASE
        if k < 0:
            raise ValueError(f"Block {block_val} < BLOCK_BASE {BLOCK_BASE}; adjust BLOCK_BASE or your data.")
        fwd_i = start_idx + 2 * k
        rev_i = fwd_i + 1
        if rev_i >= len(primers):
            raise IndexError(
                f"Not enough primers for Block {block_val}: need indices up to {rev_i} (0-based), "
                f"but only {len(primers)} primers available."
            )
        (fwd_name, fwd_seq) = primers[fwd_i]
        (rev_name, rev_seq) = primers[rev_i]
        return fwd_name, fwd_seq, rev_name, rev_seq

    df_out_blocks = []
    for blk in unique_blocks:
        fwd_name, fwd_seq, rev_name, rev_seq = primers_for_block(int(blk))
        df_block = df[df["Block"] == blk]
        df_block = apply_pair_to_block_rows(df_block, fwd_name, fwd_seq, rev_name, rev_seq, frag_cols_sorted)
        df_out_blocks.append(df_block)

    df_out = pd.concat(df_out_blocks, axis=0, ignore_index=True).sort_values(by="Block")

# =========================
# MODE 2: COMBINATORIAL
# =========================
elif PRIMER_ASSIGN_MODE == "combinatorial":
    ensure_pair_table(primers, PAIRS_CSV, ALL_PAIRS_CSV)
    pairs_df = pd.read_csv(PAIRS_CSV)

    required_pair_cols = {"Fwd_Name", "Fwd_Seq", "Rev_Name", "Rev_Seq"}
    if not required_pair_cols.issubset(set(pairs_df.columns)):
        raise ValueError(f"Pairs CSV must contain columns {required_pair_cols}. Found: {list(pairs_df.columns)}")

    if len(pairs_df) < num_blocks:
        raise ValueError(f"Not enough primer pairs: {len(pairs_df)} pairs available but {num_blocks} blocks present.")

    print(f"[INFO] Assigning {num_blocks} primer pairs to {num_blocks} blocks using first {num_blocks} rows from {PAIRS_CSV}.")

    df_out_blocks = []
    for i, blk in enumerate(unique_blocks):
        pr = pairs_df.iloc[i]
        fwd_name = str(pr["Fwd_Name"]).strip()
        fwd_seq  = str(pr["Fwd_Seq"]).strip().upper()
        rev_name = str(pr["Rev_Name"]).strip()
        rev_seq  = str(pr["Rev_Seq"]).strip().upper()

        df_block = df[df["Block"] == blk]
        df_block = apply_pair_to_block_rows(df_block, fwd_name, fwd_seq, rev_name, rev_seq, frag_cols_sorted)
        df_out_blocks.append(df_block)

    df_out = pd.concat(df_out_blocks, axis=0, ignore_index=True).sort_values(by="Block")

    if len(pairs_df) > num_blocks:
        unused_pairs_df = pairs_df.iloc[num_blocks:].reset_index(drop=True)
        unused_pairs_df.to_csv(OUTPUT_UNUSED_PAIRS, index=False)
        print(f"[INFO] Wrote unused pairs -> {OUTPUT_UNUSED_PAIRS} ({len(unused_pairs_df)} of {len(pairs_df)} remain).")
    else:
        print("[INFO] All available pairs used; no unused pairs file written.")

else:
    raise ValueError("PRIMER_ASSIGN_MODE must be 'unique_pairs' or 'combinatorial'.")

if WRITE_UNUSED_PRIMERS:
    write_unused_primers(primers, df_out, OUTPUT_UNUSED_PRIMERS)

# =========================
# OUTPUTS
# =========================

oh_cols = sorted([c for c in df_out.columns if c.startswith("Overhang")],
                 key=lambda x: int(x.replace("Overhang", ""))) if any(c.startswith("Overhang") for c in df_out.columns) else []

base_cols   = ["Block", "Length Distribution", "Sequence Name"]
primer_cols = ["Primer_Forward_Name", "Primer_Forward_Seq", "Primer_Reverse_Name", "Primer_Reverse_Seq"]
vec_cols = [c for c in ["VectorOH1", "VectorOH1_Source", "VectorOH2"] if c in df_out.columns]

col_order = base_cols + vec_cols + oh_cols + frag_cols_sorted
if "Full Sequence" in df_out.columns:
    col_order += ["Full Sequence"]
col_order += primer_cols

for c in col_order:
    if c not in df_out.columns:
        df_out[c] = ""
df_out = df_out[col_order]

df_out.to_csv(OUTPUT_WITH_PRIMERS, index=False)

if "Full Sequence" in df_out.columns:
    with open(OUTPUT_FASTA, "w") as fh:
        for _, row in df_out.iterrows():
            fh.write(f">Block_{int(row['Block'])}_{row['Sequence Name']}\n{row['Full Sequence']}\n")
    print(f"[INFO] Wrote FASTA -> {OUTPUT_FASTA}")

frag_rows = []
for _, row in df_out.iterrows():
    seq_name = row["Sequence Name"]
    for i, col in enumerate(frag_cols_sorted, start=1):
        frag = row[col]
        if is_nonempty_seq(frag):
            frag_rows.append({"Gene_Fragment": f"{seq_name}_Fragment{i}", "Sequence": str(frag)})
pd.DataFrame(frag_rows).to_csv(OUTPUT_FRAGMENTS_CSV, index=False)

print(f"[INFO] Wrote FULL_INFO -> {OUTPUT_WITH_PRIMERS}")
print(f"[INFO] Wrote fragments CSV -> {OUTPUT_FRAGMENTS_CSV}")
print("[DONE] Primer assignment complete.")


## Merging oPool order fragment csvs

In [ ]:
import pandas as pd
from pathlib import Path

if "RUN_PATHS" not in globals():
    raise RuntimeError("Run the Gene optimisation variables cell first so RUN_PATHS is defined.")

# Root directory containing run output folders/files
ROOT_DIR = RUN_PATHS["run_dir"]

# Output CSV
OUT_CSV = RUN_PATHS["merged_fragments"]

# Find all CSVs whose names end with "oPool_Order_Fragments.csv"
csv_files = sorted(
    p for p in ROOT_DIR.rglob("*oPool_Order_Fragments.csv")
    if p.resolve() != OUT_CSV.resolve()
)

print("Found files:")
for f in csv_files:
    print(" -", f)

if not csv_files:
    raise FileNotFoundError(f"No *oPool_Order_Fragments.csv files found under {ROOT_DIR}")

# Read and merge all files (they already share the same header)
dfs = [pd.read_csv(f) for f in csv_files]
merged = pd.concat(dfs, ignore_index=True)

# Write out a single CSV with one header row
merged.to_csv(OUT_CSV, index=False)

print(f"\nSaved merged CSV to:\n{OUT_CSV}")
